# Proyecto Sensores colectivos

**Objetivo:** transformar el dataset crudo `SENSORES(sucio).xlsx` en una versión limpia (`dfClean`) mediante un pipeline orientado a objetos (clase `SensoresDataCleaner`), siguiendo la fase de *Preparación de los Datos* de la metodología CRISP-DM. Se integra además la Fuente 3 (sueño) desde `sleepDay_merged.csv`.

**Requisitos de diseño:**
* **Constructor**: la clase encapsula todo el pipeline; `__init__` recibe el DataFrame original **o** la ruta del archivo y centraliza los parámetros globales (factor IQR = 1.5).
* **SRP**: cada etapa es un método independiente (diagnóstico, columnas fantasma, fechas, días sin uso, outliers, categorías, integración de sueño, validación).
* **OCP**: la regla IQR es una primitiva única (`tratarOutliersIqr`) que no cambia; la lista de columnas es configurable por constructor o por llamada.
* **Nomenclatura**: variables y métodos en `lowerCamelCase` (`dfClean`, `tratarOutliersIqr`, `limSup`, `dfSensores`). Los nombres de columnas de datos se conservan tal cual vienen en los datasets (`TotalSteps`, `SedentaryMinutes`, `SleepDay`).

## 0. Imports

In [1]:
import json
import pandas as pd
import requests

## 1. Carga de la Fuente 1 — Dataset de sensores (Excel)

In [2]:
dfSensores = pd.read_excel("SENSORES(sucio).xlsx")

In [3]:
print("Dimensiones (Filas, Columnas):", dfSensores.shape)
print("\nTipos de datos detectados:\n", dfSensores.dtypes)
print("\nPrimeros 3 registros:\n", dfSensores.head(3))

Dimensiones (Filas, Columnas): (940, 17)

Tipos de datos detectados:
 Id                            int64
ActivityDate                 object
TotalSteps                    int64
TotalDistance               float64
VeryActiveDistance          float64
ModeratelyActiveDistance    float64
LightActiveDistance         float64
SedentaryActiveDistance     float64
VeryActiveMinutes             int64
FairlyActiveMinutes           int64
LightlyActiveMinutes          int64
SedentaryMinutes              int64
Calories                      int64
Nivel_Pasos                     str
Nivel_Sedentarismo              str
Unnamed: 15                 float64
Unnamed: 16                     str
dtype: object

Primeros 3 registros:
            Id         ActivityDate  TotalSteps  TotalDistance  \
0  1503960366  2016-12-04 00:00:00       13162   8.500000e+00   
1  1503960366            4/13/2016       10735   6.970000e+14   
2  1503960366            4/14/2016       10460   6.740000e+14   

   VeryActiveDistan

## 2. Carga de la Fuente 2 — API REST OpenWeatherMap (JSON)

La API responde en formato JSON; se convierte a tabla con `pd.DataFrame(climaDatos)`. Se usa solo como contexto complementario.

In [4]:
apiKey = "d7ec6c327fa27f4f6b8441a33d7a2075"
ciudades = ["Bogota", "Medellin", "Cali", "Ibague"]
climaDatos = []

for ciudad in ciudades:
    url = f"https://api.openweathermap.org/data/2.5/weather?q={ciudad}&appid={apiKey}&units=metric&lang=es"
    try:
        res = requests.get(url)
        if res.status_code == 200:
            data = res.json()
            climaDatos.append({
                "Ciudad": data["name"],
                "Temperatura_C": data["main"]["temp"],
                "Humedad_Pct": data["main"]["humidity"],
                "Estado_Clima": data["weather"][0]["description"],
                "Velocidad_Viento": data["wind"]["speed"]
            })
        else:
            print(f"Error {res.status_code} al consultar {ciudad}")
    except Exception as e:
        print(f"Error de conexión al consultar {ciudad}: {e}")

dfApiClima = pd.DataFrame(climaDatos)
print("\n=== FUENTE 2: DATOS CLIMÁTICOS (API REST / JSON) ===")
print(dfApiClima)


=== FUENTE 2: DATOS CLIMÁTICOS (API REST / JSON) ===
             Ciudad  Temperatura_C  Humedad_Pct     Estado_Clima  \
0            Bogota          13.59           72            nubes   
1          Medellín          19.97           77            nubes   
2  Santiago de Cali          26.00           65  lluvia moderada   
3            Ibagué          25.95           42            nubes   

   Velocidad_Viento  
0              2.06  
1              1.03  
2              4.63  
3              3.09  


## 3. Carga preliminar — Sueño (CSV)

Se integra formalmente dentro de la clase (`integrarFuenteSueno`); aquí solo se inspecciona el archivo en disco `sleepDay_merged.csv`.

Observación relevante: la columna `SleepDay` viene como texto (`4/12/2016 12:00:00 AM`), por lo que el método de integración la convertirá a `datetime` y la renombrará internamente a `ActivityDate` para poder cruzar con la Fuente 1.

In [5]:
dfSleepRaw = pd.read_csv("sleepDay_merged.csv")
print("Dimensiones (Filas, Columnas):", dfSleepRaw.shape)
print("Columnas:", list(dfSleepRaw.columns))
print("\nPrimeros 3 registros:\n", dfSleepRaw.head(3))

Dimensiones (Filas, Columnas): (413, 5)
Columnas: ['Id', 'SleepDay', 'TotalSleepRecords', 'TotalMinutesAsleep', 'TotalTimeInBed']

Primeros 3 registros:
            Id               SleepDay  TotalSleepRecords  TotalMinutesAsleep  \
0  1503960366  4/12/2016 12:00:00 AM                  1                 327   
1  1503960366  4/13/2016 12:00:00 AM                  2                 384   
2  1503960366  4/15/2016 12:00:00 AM                  1                 412   

   TotalTimeInBed  
0             346  
1             407  
2             442  


---
## 4. Clase SensoresDataCleaner

La clase encapsula el pipeline completo. Se define **en una sola celda** para evitar fragmentos incompatibles.

**Diseño (SOLID):**
* `__init__` — centraliza los parámetros globales: factor IQR (por defecto 1.5), lista de columnas sujetas a outliers y lista de columnas categóricas. Acepta DataFrame **o** ruta de archivo; nunca modifica el original (`self.dfOriginal.copy()`).
* `diagnosticar` — PASO 0. Reporta dimensiones, nulos, duplicados y hallazgos visibles.
* `eliminarColumnasFantasma` — PASO 1. Borra columnas `Unnamed` creadas por Excel.
* `normalizarFechas` — PASO 2. Convierte `ActivityDate` a tipo fecha.
* `integrarFuenteSueno` — PASO INTEGRACIÓN. Carga `sleepDay_merged.csv`, depura duplicados, normaliza la clave fecha y cruza con *Left Join*.
* `filtrarDiasSinUso` — PASO 3. Elimina registros MNAR (días sin portar el tracker).
* `tratarOutliersIqr` — PASO 4. Primitiva única de la regla de Tukey (winsorización con `clip`). No conoce la lista de columnas: recibe una sola y **no cambia jamás**.
* `aplicarOutliers` — PASO 5. Orquesta `tratarOutliersIqr` sobre una lista *configurable* de columnas (desde el constructor o por llamada): extensión por configuración, sin tocar el algoritmo.
* `limpiarCategorias` — PASO 6. Normaliza cadenas con `str.strip()`.
* `validar` — PASO 7. Verifica criterios explícitos (filas, nulos, duplicados, rangos).
* `exportarCsv` — exporta el resultado sin sobrescribir el original.
* `ejecutarPipeline` — orquestador: corre las etapas en orden y devuelve `dfClean`.

In [6]:
class SensoresDataCleaner:
    """Pipeline de limpieza de datos del proyecto Sensores colectivos (CRISP-DM)."""

    def __init__(self, fuente, factorIqr=1.5, colsOutlier=None, colsCategoricas=None):
        """Recibe el DataFrame original o la ruta del archivo."""
        if isinstance(fuente, pd.DataFrame):
            self.dfOriginal = fuente.copy()
        else:
            self.dfOriginal = pd.read_excel(fuente)
        self.factorIqr = factorIqr
        self.colsOutlier = colsOutlier if colsOutlier is not None else \
            ["TotalSteps", "Calories", "VeryActiveMinutes"]
        self.colsCategoricas = colsCategoricas if colsCategoricas is not None else \
            ["Nivel_Pasos", "Nivel_Sedentarismo"]
        self.colFecha = "ActivityDate"
        self.dfClean = None

    def diagnosticar(self):
        print("=== PASO 0: DIAGNÓSTICO INICIAL DE CALIDAD ===")
        print("Dimensiones:", self.dfOriginal.shape)
        print("Nulos explícitos (isna):", self.dfOriginal.isna().sum().sum())
        print("Duplicados exactos:", self.dfOriginal.duplicated().sum())
        print("\nNulos por columna:")
        print(self.dfOriginal.isna().sum()[lambda s: s > 0])
        colsFantasma = [c for c in self.dfOriginal.columns if "Unnamed" in c]
        print("\nColumnas fantasma (artefactos de Excel):", colsFantasma)
        diasSinUso = ((self.dfOriginal["TotalSteps"] == 0) &
                     (self.dfOriginal["SedentaryMinutes"] == 1440)).sum()
        print(f"\nDías de 'no uso' del tracker (0 pasos y 1440 min): {diasSinUso}")
        return self

    def eliminarColumnasFantasma(self):
        colsFantasma = [c for c in self.dfClean.columns if "Unnamed" in c]
        self.dfClean = self.dfClean.drop(columns=colsFantasma).copy()
        print(f"[PASO 1] Columnas fantasma eliminadas: {colsFantasma}")
        print(f"Nuevas dimensiones: {self.dfClean.shape}")
        return self

    def normalizarFechas(self):
        self.dfClean[self.colFecha] = pd.to_datetime(
            self.dfClean[self.colFecha], errors="coerce")
        fechasInvalidas = self.dfClean[self.colFecha].isna().sum()
        print(f"[PASO 2] '{self.colFecha}' convertida a datetime.")
        print(f"Fechas inválidas (NaT): {fechasInvalidas}")
        return self

    def integrarFuenteSueno(self, rutaSleep="sleepDay_merged.csv"):
        """PASO INTEGRACIÓN: Carga, depura e integra la Fuente 3 (Sueño)."""
        # 1. Carga de la Fuente 3
        dfSleep = pd.read_csv(rutaSleep)
        filasIniciales = len(dfSleep)

        # 2. Eliminación de duplicados exactos en origen
        dfSleep = dfSleep.drop_duplicates(subset=["Id", "SleepDay"]).copy()
        duplicadosEliminados = filasIniciales - len(dfSleep)

        # 3. Normalización de la fecha clave
        dfSleep["ActivityDate"] = pd.to_datetime(dfSleep["SleepDay"], format="mixed")

        # 4. Feature Engineering con control de rango [0.0 - 1.0]
        dfSleep["Eficiencia_Sueño"] = (
            dfSleep["TotalMinutesAsleep"] / dfSleep["TotalTimeInBed"]
        ).round(2).clip(upper=1.0)

        # 5. Cruce de datos (Left Join)
        self.dfClean = pd.merge(
            self.dfClean,
            dfSleep[["Id", "ActivityDate", "TotalMinutesAsleep", "TotalTimeInBed", "Eficiencia_Sueño"]],
            on=["Id", "ActivityDate"],
            how="left"
        )
        print(f"[PASO SUEÑO] Registros limpios de sueño. Duplicados eliminados: {duplicadosEliminados}")
        print(f"[PASO INTEGRACIÓN] Columnas consolidadas: {self.dfClean.shape[1]}")
        return self

    def filtrarDiasSinUso(self):
        indicesNoUso = self.dfClean[
            (self.dfClean["TotalSteps"] == 0) &
            (self.dfClean["SedentaryMinutes"] == 1440)
        ].index
        self.dfClean = self.dfClean.drop(index=indicesNoUso).reset_index(drop=True)
        print(f"[PASO 3] Registros de 'No Uso' eliminados: {len(indicesNoUso)}")
        print(f"Filas restantes: {len(self.dfClean)}")
        return self

    def tratarOutliersIqr(self, columna):
        """Regla de Tukey: winsoriza una sola columna. Primitiva inmutable (OCP)."""
        q1 = self.dfClean[columna].quantile(0.25)
        q3 = self.dfClean[columna].quantile(0.75)
        iqr = q3 - q1
        limInf = q1 - self.factorIqr * iqr
        limSup = q3 + self.factorIqr * iqr
        limInfReal = max(0, limInf)
        fuera = self.dfClean.loc[
            (self.dfClean[columna] < limInf) | (self.dfClean[columna] > limSup)
        ].shape[0]
        self.dfClean[columna] = self.dfClean[columna].clip(
            lower=limInfReal, upper=limSup)
        print(f"  -> '{columna}': IQR={iqr:.2f} | "
              f"límites=[{limInfReal:.2f}, {limSup:.2f}] | "
              f"outliers recortados={fuera}")
        return self

    def aplicarOutliers(self, cols=None):
        """Aplica la regla IQR a una lista configurable de columnas (OCP):
        la lista se puede pasar aquí o en el constructor; el algoritmo no cambia."""
        colsAplicar = self.colsOutlier if cols is None else cols
        print("[PASO 5] Aplicando regla IQR (winsorización):")
        for col in colsAplicar:
            self.tratarOutliersIqr(col)
        return self

    def limpiarCategorias(self, cols=None):
        colsAplicar = self.colsCategoricas if cols is None else cols
        for col in colsAplicar:
            self.dfClean[col] = self.dfClean[col].str.strip()
            print(f"[PASO 6] '{col}' -> valores únicos: "
                  f"{sorted(self.dfClean[col].unique())}")
        return self

    def validar(self):
        print("=== VALIDACIÓN FINAL (dfClean) ===")
        print(f"Dimensiones: {self.dfClean.shape}")
        print(f"Nulos totales: {self.dfClean.isna().sum().sum()}")
        print(f"Duplicados exactos: {self.dfClean.duplicated().sum()}")
        print("\nEstadísticas finales de las métricas de actividad:")
        print(self.dfClean[["TotalSteps", "Calories", "VeryActiveMinutes"]].describe().round(1))
        return self

    def exportarCsv(self, ruta="Sensores_limpio.csv"):
        self.dfClean.to_csv(ruta, index=False)
        print(f"Dataset limpio exportado: {ruta} ({self.dfClean.shape[0]} filas)")
        return self

    def ejecutarPipeline(self):
        """Orquestador: corre las etapas en orden y devuelve dfClean."""
        self.dfClean = self.dfOriginal.copy()
        self.diagnosticar()
        self.eliminarColumnasFantasma()
        self.normalizarFechas()
        self.integrarFuenteSueno()
        self.filtrarDiasSinUso()
        self.aplicarOutliers()
        self.limpiarCategorias()
        self.validar()
        return self.dfClean

## 5. Instanciar y ejecutar el pipeline

Se crea un `SensoresDataCleaner` pasándole el DataFrame original y el factor IQR; luego `ejecutarPipeline()` corre todas las etapas (incluida la integración de la Fuente 3 justo después de normalizar fechas, para que ambas claves coincidan en `datetime`) y retorna `dfClean`. El original `dfSensores` nunca se modifica.

In [7]:
limpiador = SensoresDataCleaner(dfSensores, factorIqr=1.5)
dfClean = limpiador.ejecutarPipeline()

=== PASO 0: DIAGNÓSTICO INICIAL DE CALIDAD ===
Dimensiones: (940, 17)
Nulos explícitos (isna): 1879
Duplicados exactos: 0

Nulos por columna:
Unnamed: 15    940
Unnamed: 16    939
dtype: int64

Columnas fantasma (artefactos de Excel): ['Unnamed: 15', 'Unnamed: 16']

Días de 'no uso' del tracker (0 pasos y 1440 min): 72
[PASO 1] Columnas fantasma eliminadas: ['Unnamed: 15', 'Unnamed: 16']
Nuevas dimensiones: (940, 15)
[PASO 2] 'ActivityDate' convertida a datetime.
Fechas inválidas (NaT): 0
[PASO SUEÑO] Registros limpios de sueño. Duplicados eliminados: 3
[PASO INTEGRACIÓN] Columnas consolidadas: 18
[PASO 3] Registros de 'No Uso' eliminados: 72
Filas restantes: 868
[PASO 5] Aplicando regla IQR (winsorización):
  -> 'TotalSteps': IQR=6230.25 | límites=[0.00, 20400.38] | outliers recortados=15
  -> 'Calories': IQR=976.75 | límites=[388.62, 4295.62] | outliers recortados=11
  -> 'VeryActiveMinutes': IQR=35.00 | límites=[0.00, 87.50] | outliers recortados=56
[PASO 6] 'Nivel_Pasos' -> valores

### 6. Verificación de OCP: columnas configurables sin tocar el algoritmo

La lista de columnas sujetas a outliers se modifica **por configuración** (constructor), no por cambios en la primitiva `tratarOutliersIqr`.

In [8]:
limpiadorAlt = SensoresDataCleaner("SENSORES(sucio).xlsx",
                                 colsOutlier=["TotalSteps", "TotalDistance"])
print("Configuración OCP -> colsOutlier:", limpiadorAlt.colsOutlier)
print("Configuración OCP -> factorIqr  :", limpiadorAlt.factorIqr)

Configuración OCP -> colsOutlier: ['TotalSteps', 'TotalDistance']
Configuración OCP -> factorIqr  : 1.5


In [9]:
limpiador.exportarCsv()

Dataset limpio exportado: Sensores_limpio.csv (868 filas)


## 5. Bitácora de limpieza

| Paso | Método | Acción Realizada | Impacto / Métricas en el Dataset |
|:---:|---|---|---|
| **0** | `diagnosticar` | Inspección general del dataset crudo con `isna()` y `duplicated()`. | **Inicio:** 940 filas × 17 cols. 1879 nulos (exclusivos de cols fantasma), 0 duplicados. |
| **1** | `eliminarColumnasFantasma` | Eliminación de columnas `Unnamed` generadas por artefactos de Excel. | **-2 columnas** de artefactos (pasa a 15 cols). |
| **2** | `normalizarFechas` | Conversión de la columna `ActivityDate` a `datetime64` con `errors="coerce"`. | 0 fechas inválidas / NaT generados. |
| **3** | `integrarFuenteSueno` | Carga de `sleepDay_merged.csv`, deduplicación por `(Id, SleepDay)`, parseo a `datetime` y *Left Join*. | **-3 duplicados** en origen de sueño; **+3 columnas** (`TotalMinutesAsleep`, `TotalTimeInBed`, `Eficiencia_Sueño`). |
| **4** | `filtrarDiasSinUso` | Eliminación de registros con 0 pasos y 1440 min sedentarios (mecanismo MNAR). | **-72 filas** de días no portados. |
| **5** | `tratarOutliersIqr` / `aplicarOutliers` | Winsorización con `clip()` siguiendo la regla de Tukey ($1.5 \times \text{IQR}$). | Recorte acotado en `TotalSteps`, `Calories` y `VeryActiveMinutes`. |
| **6** | `limpiarCategorias` | Normalización de cadenas de texto mediante `str.strip()`. | Estandarización de `Nivel_Pasos` y `Nivel_Sedentarismo`. |
| **7** | `validar` / `exportarCsv` | Verificación final de integridad de datos y exportación a `Sensores_limpio.csv`. | **Final:** 868 filas × 18 cols, 0 duplicados, 0 nulos en métricas de sensores. |

---

### Notas Técnicas y Hallazgos para la Fase de Negocio
- **Observación en `TotalDistance`:** Presenta magnitudes anómalas del orden de $10^{14}$ (error sistemático de la fuente). La winsorización IQR no aplica por no tratarse de outliers aislados; requiere validación con la fuente de origen.
- **Cobertura de la Fuente 3:** El cruce mediante *Left Join* mantiene la totalidad de los 868 registros válidos de actividad física, dejando valores `NaN` únicamente en los días donde el usuario no registró monitoreo de sueño.